In [ ]:
import timm
from huggingface_hub import hf_hub_download
import torch
import torch.nn as nn

# Define model details
model_name_or_path = "edadaltocg/resnet50_cifar10"
file_name = "pytorch_model.bin"

# Download the checkpoint from Hugging Face Hub
checkpoint_path = hf_hub_download(repo_id=model_name_or_path, filename=file_name)

# Create a standard ResNet50 model
# We will manually modify the stem for CIFAR
model = timm.create_model(
    'resnet50',
    pretrained=False,
    num_classes=10  # Set num_classes for the final layer
)

# Manually modify the stem for CIFAR-10:
# 1. Change conv1 from 7x7 stride 2 to 3x3 stride 1
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
# 2. Replace maxpool with Identity (no pooling)
model.maxpool = nn.Identity()

# Load the state_dict
state_dict = torch.load(checkpoint_path, map_location='cpu')

# The checkpoint might contain a 'model_state_dict' key if it's from a trainer
# Or it might be the raw state_dict. Check the structure.
if 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']

# Load the state dict into the model
# Use strict=True for exact matching, if there are minor mismatches, strict=False can be used
model.load_state_dict(state_dict, strict=True)

print(model)
print("Model loaded successfully!")

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act2): ReLU(inplace=True)
      (aa): Identity()
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act3): ReLU(inplace=True)
      (downsample): Sequential(
    

In [ ]:
import torch.nn as nn

# Define model details
model_name_or_path = "edadaltocg/resnet50_cifar10"
file_name = "pytorch_model.bin"

# Download the checkpoint from Hugging Face Hub
checkpoint_path = hf_hub_download(repo_id=model_name_or_path, filename=file_name)

# Create a standard ResNet50 model
# We will manually modify the stem for CIFAR
model = timm.create_model(
    'resnet50',
    pretrained=False,
    num_classes=10  # Set num_classes for the final layer
)

# Manually modify the stem for CIFAR-10:
# 1. Change conv1 from 7x7 stride 2 to 3x3 stride 1
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
# 2. Replace maxpool with Identity (no pooling)
model.maxpool = nn.Identity()

# Load the state_dict
state_dict = torch.load(checkpoint_path, map_location='cpu')

# The checkpoint might contain a 'model_state_dict' key if it's from a trainer
# Or it might be the raw state_dict. Check the structure.
if 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']

# Load the state dict into the model
# Use strict=True for exact matching, if there are minor mismatches, strict=False can be used
model.load_state_dict(state_dict, strict=True)

print(model)
print("Model loaded successfully!")

In [2]:
import torchvision
import torchvision.transforms as transforms
import torch

# Define the transformations for the CIFAR-10 test set
# These transformations should match the ones used during training if possible,
# but at minimum include ToTensor and normalization appropriate for the model.
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load the CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)

# Create a DataLoader for the test set
batch_size = 100 # You can adjust this batch size
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

# Define the CIFAR-10 class names for later use
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print("CIFAR-10 test dataset loaded successfully.")
print(f"Number of test samples: {len(testset)}")

100%|██████████| 170M/170M [00:15<00:00, 11.3MB/s] 


CIFAR-10 test dataset loaded successfully.
Number of test samples: 10000


In [3]:
# Set the model to evaluation mode
model.eval()

correct = 0
total = 0

# Disable gradient calculations during inference
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # If you have a GPU, move data to GPU
        # images = images.to('cuda')
        # labels = labels.to('cuda')

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the model on the 10000 test images: {accuracy:.2f}%')

Accuracy of the model on the 10000 test images: 94.65%
